In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


In [3]:
kr_clean_files = [
    "Ethiopia_KR_2008_clean.csv",
    "Ethiopia_KR_2011_clean.csv",
    "Kenya_KR_2003_clean.csv",
    "Kenya_KR_2009_clean.csv",
    "Kenya_KR_2014_clean.csv",
    "Kenya_KR_2022_clean.csv",
    "Tanzania_KR_2010_clean.csv",
    "Tanzania_KR_2015_clean.csv",
    "Tanzania_KR_2022_clean.csv",
    "Uganda_KR_2011_clean.csv",
    "Uganda_KR_2016_clean.csv",
]


In [5]:
children_list = []

for fname in kr_clean_files:
    print("reading", fname)
    df = pd.read_csv(fname)
    cols = df.columns.tolist()
    
    # figure out country from file name
    country = fname.split("_")[0]
    df["country"] = country
    
    # try to get survey year
    year_col = pick_col(cols, ["interview year", "year_interview", "int_year", "v007"])
    if year_col is None:
        print("  could not find survey year, skipping this file for now")
        continue
    df["survey_year"] = df[year_col]
    
    # child age in months
    age_col = pick_col(
        cols,
        ["child_age_months", "age in months", "b19"]
    )
    
    # anthropometry z scores (height for age etc)
    haz_col = pick_col(
        cols,
        ["haz_height_for_age_z", "height for age SD per WHO", "z_height_for_age", "hw70"]
    )
    waz_col = pick_col(
        cols,
        ["waz_weight_for_age_z", "weight for age SD per WHO", "z_weight_for_age", "hw71"]
    )
    whz_col = pick_col(
        cols,
        ["whz_weight_for_height_z", "weight for height SD per WHO", "z_weight_for_height", "hw72"]
    )
    
    # simple econ or context variables
    wealth_col = pick_col(cols, ["wealth_quintile", "wealth index combined"])
    edu_col = pick_col(cols, ["mother_edu_level", "highest education level"])
    urban_col = pick_col(cols, ["urban_rural", "v025"])
    
    # build a small table with only what I care about for now
    keep_cols = ["country", "survey_year"]
    
    if age_col is not None:
        keep_cols.append(age_col)
    if haz_col is not None:
        keep_cols.append(haz_col)
    if waz_col is not None:
        keep_cols.append(waz_col)
    if whz_col is not None:
        keep_cols.append(whz_col)
    if wealth_col is not None:
        keep_cols.append(wealth_col)
    if edu_col is not None:
        keep_cols.append(edu_col)
    if urban_col is not None:
        keep_cols.append(urban_col)
    
    small = df[keep_cols].copy()
    
    # rename columns to a common set so life is easier later
    rename_map = {}
    if age_col is not None:
        rename_map[age_col] = "child_age_months"
    if haz_col is not None:
        rename_map[haz_col] = "haz_height_for_age_z"
    if waz_col is not None:
        rename_map[waz_col] = "waz_weight_for_age_z"
    if whz_col is not None:
        rename_map[whz_col] = "whz_weight_for_height_z"
    if wealth_col is not None:
        rename_map[wealth_col] = "wealth_quintile"
    if edu_col is not None:
        rename_map[edu_col] = "mother_edu_level"
    if urban_col is not None:
        rename_map[urban_col] = "urban_rural"
    
    small = small.rename(columns=rename_map)
    
    children_list.append(small)

# put them together
children_all = pd.concat(children_list, ignore_index=True)

children_all.head()


reading Ethiopia_KR_2008_clean.csv


NameError: name 'pick_col' is not defined